# ☁️ Etapa 3: Model Registry y Versionado en Producción (MinIO)

Este notebook representa el eslabón final del pipeline de **MLOps**. Su función es actuar como un **Model Registry** simplificado, encargándose de organizar, versionar y persistir el modelo optimizado en el almacenamiento de objetos de grado de producción.

---

### 📂 1. Gestión del Espacio de Trabajo
Para garantizar la estabilidad en **OpenShift AI**, el notebook:
* **Fuerza el Directorio de Trabajo:** Establece `/opt/app-root/src` como base, evitando conflictos de permisos de escritura y errores de metadatos (`utime`) propios de los entornos de Kubernetes.
* **Extracción de Artefactos:** Recupera el paquete `modelo_onnx.tar.gz` generado por el nodo anterior del pipeline.

### 🔢 2. Inteligencia de Versionado (v+1)
El script implementa una lógica de despliegue no destructivo:
1. **Inspección de S3:** Conecta con el bucket `modelos-prod` y lista las carpetas existentes.
2. **Detección de Versión:** Mediante **Expresiones Regulares (Regex)**, identifica el número de versión más alto (ej: `v4`).
3. **Incremento Automático:** Calcula la nueva ruta (ej: `v5`) para asegurar que cada ejecución del pipeline sea trazable y permita realizar un *Rollback* si es necesario.

### 🚀 3. Carga y Registro de Despliegue
* **Transferencia:** Sube de forma iterativa todos los componentes del modelo ONNX (pesos, configuración y tokenizador) a la nueva carpeta versionada en **MinIO**.
* **Archivo de Control (`deployed_version.txt`):** Este es el paso más importante para la automatización. El notebook genera un archivo de texto pequeño que contiene únicamente el nombre de la versión (ej: `v5`). Este archivo sirve como "testigo" para que otros procesos o el administrador de la plataforma sepan exactamente qué versión se acaba de procesar.

### 🛡️ 4. Resiliencia ante Fallos
El código incluye un bloque de manejo de excepciones que, en caso de error, genera un archivo con la palabra `error`. Esto evita que el pipeline de **Elyra** falle por archivos no encontrados y permite que el operador pueda auditar el log de errores dentro de la plataforma de forma controlada.

---
> **Estado:** Al finalizar este notebook, el modelo está oficialmente "registrado" en la nube de **Movistar** y listo para ser servido por el **InferenceService**.

In [ ]:
import os
import boto3
import tarfile
import re
from botocore.client import Config

# --- CONFIGURACIÓN DE RUTAS ---
# Forzamos el directorio de trabajo para evitar el error de 'utime'
WORKING_DIR = "/opt/app-root/src"
os.chdir(WORKING_DIR)

input_tar = "modelo_onnx.tar.gz"
output_info = "deployed_version.txt"
onnx_path = "modelo_t-moviles_onnx"

print(f"📂 Iniciando proceso en {WORKING_DIR}...")

try:
    # 1. Extraer el paquete ONNX
    if os.path.exists(input_tar):
        print(f"📦 Extrayendo {input_tar}...")
        with tarfile.open(input_tar, "r:gz") as tar:
            tar.extractall(path=".")
    else:
        print("⚠️ El archivo modelo_onnx.tar.gz no se encontró. ¿Es el primer run?")

    # 2. Conexión S3
    s3 = boto3.client('s3', 
                      endpoint_url=os.getenv("S3_ENDPOINT", "http://minio-service.t-moviles-ai.svc.cluster.local:9000"),
                      aws_access_key_id="minio", 
                      aws_secret_access_key="minio123",
                      config=Config(signature_version='s3v4'), 
                      verify=False)

    # 3. Lógica de Versionado
    bucket = "modelos-prod"
    res = s3.list_objects_v2(Bucket=bucket, Delimiter='/')
    prefixes = [p['Prefix'] for p in res.get('CommonPrefixes', [])]
    versions = [int(re.search(r'v(\d+)', p).group(1)) for p in prefixes if re.search(r'v(\d+)', p)]
    new_v = f"v{max(versions + [0]) + 1}"

    # 4. Subida de archivos
    print(f"🚀 Subiendo archivos a MinIO carpeta: {new_v}")
    if os.path.exists(onnx_path):
        for file in os.listdir(onnx_path):
            s3.upload_file(os.path.join(onnx_path, file), bucket, f"{new_v}/{file}")
    else:
        # Fallback por si la carpeta tiene otro nombre dentro del tar
        print(f"⚠️ No se encontró {onnx_path}, revisando contenido...")
        print(os.listdir("."))
        new_v = "v1-fallback" # Para que el pipeline no muera sin el archivo de salida

    # 5. GENERAR EL ARCHIVO DE SALIDA (Crucial para Elyra)
    # Se guarda en WORKING_DIR para que Elyra lo encuentre
    with open(output_info, "w") as f:
        f.write(new_v)
    
    print(f"✅ Versión {new_v} registrada en {output_info}")

except Exception as e:
    print(f"❌ Error crítico: {e}")
    # Generamos un archivo vacío para que Elyra no falle por 'FileNotFound'
    # y podamos ver el log del error real en el output del nodo.
    with open(output_info, "w") as f:
        f.write("error")
    raise e